# Understand how Kalshi and Polymarket keep data

## Problem statement

Cross-platform arbitrage in prediction markets means finding the same real-world event listed on two different platforms when the prices do not agree. If Kalshi and Polymarket are describing the same outcome in different words, that price gap can be a signal.

The hard part is that there is no shared event ID, and the two platforms often phrase the same market very differently. This notebook is about turning that messy, mismatched text into comparable signals.

## 0: Import dependencies

In [75]:
import requests
from sentence_transformers import SentenceTransformer
from sentence_transformers import util
from dotenv import load_dotenv
from groq import Groq
import os

load_dotenv()
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

## 1: Data sources

This notebook uses Kalshi and Polymarket market-discovery APIs directly. The important finding is that both platforms expose the market data we need publicly, without authentication, for discovery and analysis. Authentication only matters later if you want to place trades.

Docs used:
- Kalshi API docs: https://docs.kalshi.com/
- Polymarket API docs: https://docs.polymarket.com/

The next cells focus on politics markets so the matching problem stays concrete and easy to inspect.

### Category and tag filtering was not obvious

The first instinct was wrong in two places:
- Kalshi category filtering belongs on `/events`, not `/markets`.
- Polymarket politics filtering uses a numeric `tag_id`, not a text category name.

The working version below uses `category == "Politics"` on the Kalshi event feed and `tag_id=2` for Polymarket politics markets.

In [76]:
kalshi_market_raw = requests.get(
    "https://api.elections.kalshi.com/trade-api/v2/markets",
    params={"status": "open", "limit": 25},
).json()["markets"]

combo_markets = [m for m in kalshi_market_raw if m.get("mve_selected_legs")]
clean_markets = [m for m in kalshi_market_raw if not m.get("mve_selected_legs")]

print(f"Raw Kalshi markets: {len(kalshi_market_raw)}")
print(f"Clean Kalshi markets after filtering combos: {len(clean_markets)}")
print(f"Combo / multivariate markets filtered out: {len(combo_markets)}\n")

if combo_markets:
    example = combo_markets[0]
    print("Example combo market before filter:")
    print(f"- {example['title']}")
    print(f"- mve_selected_legs: {example['mve_selected_legs']}\n")
    print("After filtering, that market is removed from the matching pool.")
else:
    print("No combo markets showed up in this sample, but the filter is still applied.")

Raw Kalshi markets: 25
Clean Kalshi markets after filtering combos: 0
Combo / multivariate markets filtered out: 25

Example combo market before filter:
- yes Jon Rahm,yes Tommy Fleetwood
- mve_selected_legs: [{'event_ticker': 'KXPGATOP10-THOC26', 'market_ticker': 'KXPGATOP10-THOC26-JRAH', 'side': 'yes'}, {'event_ticker': 'KXPGATOP20-THOC26', 'market_ticker': 'KXPGATOP20-THOC26-TFLE', 'side': 'yes'}]

After filtering, that market is removed from the matching pool.


## 2: The messy data problem

The raw market feeds are not clean enough to use as-is. Kalshi has combo / multivariate structures that can look like ordinary markets at first glance, but they should be filtered out before semantic matching.

This section shows the raw data first, then the cleaned version so the difference is visible.

In [77]:
kalshi_events = requests.get(
    "https://api.elections.kalshi.com/trade-api/v2/events",
    params={"status": "open", "limit": 20, "with_nested_markets": "true"},
).json()["events"]

politics_events = [e for e in kalshi_events if e.get("category") == "Politics"]

kalshi_titles = []
for e in politics_events:
    for m in e.get("markets", []):
        kalshi_titles.append(m["title"])


In [78]:
poly_events = requests.get(
    "https://gamma-api.polymarket.com/events",
    params={"tag_id": 2, "active": "true", "closed": "false", "limit": 20},
).json()

poly_titles = []
for e in poly_events:
    for m in e.get("markets", []):
        poly_titles.append(m["question"])


In [79]:
print(f"Kalshi politics titles: {len(kalshi_titles)}")
for title in kalshi_titles[:5]:
    print(f"- {title}")


Kalshi politics titles: 41
- Will Ben Wikler be the next Chair of the Democratic National Committee?
- Will Jane Kleeb be the next Chair of the Democratic National Committee?
- Will Malcolm Kenyatta be the next Chair of the Democratic National Committee?
- Will Shasti Conrad be the next Chair of the Democratic National Committee?
- Will Reyna Walters-Morgan be the next Chair of the Democratic National Committee?


In [80]:
print(f"Polymarket politics titles: {len(poly_titles)}")
for title in poly_titles[:5]:
    print(f"- {title}")


Polymarket politics titles: 475
- Macron out in 2025?
- Macron out by July 31, 2026?
- Macron out by June 30, 2026?
- Macron out by October 31, 2025?
- China x India military clash by June 30?


## 4: Embed titles with a shared encoder

This is a shared-encoder plus ai pipeline: one sentence-transformer maps both Kalshi and Polymarket titles into the same vector space, then cosine similarity compares them. 

In [81]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4605.00it/s]


In [82]:
kalshi_embeddings = model.encode(kalshi_titles)
poly_embeddings = model.encode(poly_titles)

In [83]:
print(f"Shape of Kalshi embeddings: {kalshi_embeddings.shape}")
print(f"Shape of Polymarket embeddings: {poly_embeddings.shape}\n")

print("Example — first Kalshi title and its vector (first 10 dims):")
print(f"Title: {kalshi_titles[0]}")
print(f"Vector: {kalshi_embeddings[0][:10]} ... ({len(kalshi_embeddings[0])} dims total)")


Shape of Kalshi embeddings: (41, 384)
Shape of Polymarket embeddings: (475, 384)

Example — first Kalshi title and its vector (first 10 dims):
Title: Will Ben Wikler be the next Chair of the Democratic National Committee?
Vector: [ 0.01154174 -0.0441025  -0.0077846   0.00128403 -0.02114218 -0.06508203
  0.02935044 -0.0512617   0.00403226 -0.0089955 ] ... (384 dims total)


## 5: Compare cosine similarity

In [84]:
cosine_scores = util.cos_sim(kalshi_embeddings, poly_embeddings)

In [85]:
print(f"Similarity matrix shape: {cosine_scores.shape}")

Similarity matrix shape: torch.Size([41, 475])


In [86]:
THRESHOLD = 0.5

print("\n=== Best match per Kalshi market ===\n")
for i, k_title in enumerate(kalshi_titles):
    best_idx = cosine_scores[i].argmax().item()
    best_score = cosine_scores[i][best_idx].item()
    p_title = poly_titles[best_idx]

    flag = "✅ POSSIBLE MATCH" if best_score > THRESHOLD else "—"
    print(f"[{best_score:.3f}] {flag}")
    print(f"  Kalshi:     {k_title}")
    print(f"  Polymarket: {p_title}\n")


=== Best match per Kalshi market ===

[0.645] ✅ POSSIBLE MATCH
  Kalshi:     Will Ben Wikler be the next Chair of the Democratic National Committee?
  Polymarket: Will Gavin Newsom win the 2028 Democratic presidential nomination?

[0.689] ✅ POSSIBLE MATCH
  Kalshi:     Will Jane Kleeb be the next Chair of the Democratic National Committee?
  Polymarket: Will Gina Raimondo win the 2028 Democratic presidential nomination?

[0.641] ✅ POSSIBLE MATCH
  Kalshi:     Will Malcolm Kenyatta be the next Chair of the Democratic National Committee?
  Polymarket: Will Barack Obama win the 2028 Democratic presidential nomination?

[0.649] ✅ POSSIBLE MATCH
  Kalshi:     Will Shasti Conrad be the next Chair of the Democratic National Committee?
  Polymarket: Will Ruben Gallego win the 2028 Democratic presidential nomination?

[0.671] ✅ POSSIBLE MATCH
  Kalshi:     Will Reyna Walters-Morgan be the next Chair of the Democratic National Committee?
  Polymarket: Will Liz Cheney win the 2028 Democratic pre

## 6: Trial: top cross-market matches

In [87]:
# Show the strongest matches across all Kalshi and Polymarket politics titles.
matches = []
for i in range(len(kalshi_titles)):
    for j in range(len(poly_titles)):
        matches.append((cosine_scores[i][j].item(), kalshi_titles[i], poly_titles[j]))

matches.sort(reverse=True, key=lambda x: x[0])

print("=== Top 10 matches across ALL pairs ===\n")
for score, k_title, p_title in matches[:10]:
    print(f"[{score:.3f}]")
    print(f"  Kalshi:     {k_title}")
    print(f"  Polymarket: {p_title}\n")


=== Top 10 matches across ALL pairs ===

[0.851]
  Kalshi:     Will Gretchen Whitmer be the next Chair of the Democratic National Committee?
  Polymarket: Will Gretchen Whitmer win the 2028 Democratic presidential nomination?

[0.849]
  Kalshi:     Will Rahm Emanuel be the next Chair of the Democratic National Committee?
  Polymarket: Will Rahm Emanuel win the 2028 Democratic presidential nomination?

[0.829]
  Kalshi:     Will Pete Buttigieg be the next Chair of the Democratic National Committee?
  Polymarket: Will Pete Buttigieg win the 2028 Democratic presidential nomination?

[0.823]
  Kalshi:     Will Tim Walz be the next Chair of the Democratic National Committee?
  Polymarket: Will Tim Walz win the 2028 Democratic presidential nomination?

[0.822]
  Kalshi:     Will Andy Beshear be the next Chair of the Democratic National Committee?
  Polymarket: Will Andy Beshear win the 2028 Democratic presidential nomination?

[0.817]
  Kalshi:     Will J.B. Pritzker be the next Chair of the

In [88]:
whitmer_example = next(
    (
        (score, k_title, p_title)
        for score, k_title, p_title in matches
        if "Whitmer" in k_title and "Whitmer" in p_title
    ),
    None,
)

if whitmer_example:
    score, k_title, p_title = whitmer_example
    print(f"[{score:.3f}] False positive example")
    print(f"  Kalshi:     {k_title}")
    print(f"  Polymarket: {p_title}")
else:
    print("Whitmer example not found in the current matches list.")

[0.851] False positive example
  Kalshi:     Will Gretchen Whitmer be the next Chair of the Democratic National Committee?
  Polymarket: Will Gretchen Whitmer win the 2028 Democratic presidential nomination?


## 7: Next step: add a verification layer

Cosine similarity is a good first pass, but the Whitmer example shows why it is not enough on its own. Similar names and related political topics can look like a match even when the actual outcomes are different.

The next step is a second-pass verifier, such as a Groq LLM check, that asks whether two questions are betting on the exact same real-world outcome. That extra pass should reduce false positives before anything is treated as a real arbitrage candidate.

In [89]:
def verify_match(kalshi_title, poly_title):
    """
    Ask an LLM whether two prediction market questions are betting on the
    exact same real-world outcome (not just a similar topic).
    """
    if groq_client is None:
        raise RuntimeError("Set GROQ_API_KEY before running verification cells.")

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{
            "role": "user",
            "content": f"""Are these two prediction market questions betting on the EXACT same real-world outcome? Not just a similar topic or the same person — the actual outcome being bet on must be identical.

Question A (Kalshi): {kalshi_title}
Question B (Polymarket): {poly_title}

Respond in exactly this format:
MATCH: yes or no
REASON: one short sentence"""
        }],
        temperature=0,
    )
    text = response.choices[0].message.content
    is_match = "yes" in text.split("REASON:")[0].lower()
    reason = text.split("REASON:")[-1].strip() if "REASON:" in text else text.strip()
    return is_match, reason

In [90]:
TOP_N = 50
top_candidates = matches[:TOP_N]

print(f"Verifying top {TOP_N} cosine matches with Groq...\n")

verified_results = []
for score, k_title, p_title in top_candidates:
    is_match, reason = verify_match(k_title, p_title)
    verified_results.append((score, k_title, p_title, is_match, reason))

    status = "CONFIRMED MATCH" if is_match else "FALSE POSITIVE"
    print(f"[{score:.3f}] {status}")
    print(f"  Kalshi:     {k_title}")
    print(f"  Polymarket: {p_title}")
    print(f"  Reason:     {reason}\n")

Verifying top 50 cosine matches with Groq...

[0.851] FALSE POSITIVE
  Kalshi:     Will Gretchen Whitmer be the next Chair of the Democratic National Committee?
  Polymarket: Will Gretchen Whitmer win the 2028 Democratic presidential nomination?
  Reason:     The outcome being bet on is Gretchen Whitmer becoming Chair of the Democratic National Committee versus winning the 2028 Democratic presidential nomination, which are two distinct roles.

[0.849] FALSE POSITIVE
  Kalshi:     Will Rahm Emanuel be the next Chair of the Democratic National Committee?
  Polymarket: Will Rahm Emanuel win the 2028 Democratic presidential nomination?
  Reason:     The outcome being bet on is Rahm Emanuel becoming the Chair of the Democratic National Committee versus Rahm Emanuel winning the 2028 Democratic presidential nomination, which are two distinct roles.

[0.829] FALSE POSITIVE
  Kalshi:     Will Pete Buttigieg be the next Chair of the Democratic National Committee?
  Polymarket: Will Pete Buttigie

In [91]:
def verify_match(kalshi_title, poly_title):
    """
    Ask an LLM whether two prediction market questions are betting on the
    exact same real-world outcome (not just a similar topic).
    """
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{
            "role": "user",
            "content": f"""Are these two prediction market questions betting on the EXACT same real-world outcome? Not just a similar topic or the same person — the actual outcome being bet on must be identical.

Question A (Kalshi): {kalshi_title}
Question B (Polymarket): {poly_title}

Respond in exactly this format:
MATCH: yes or no
REASON: one short sentence"""
        }],
        temperature=0,
    )
    text = response.choices[0].message.content
    is_match = "yes" in text.split("REASON:")[0].lower()
    reason = text.split("REASON:")[-1].strip() if "REASON:" in text else text.strip()
    return is_match, reason

## 8: Trying a different category — Fed rate decisions

The DNC Chair race produced zero true positives across 50 candidate pairs, because the dataset itself created a large false-positive cluster: many candidates on one side, competing against a structurally similar but distinct race on the other side. To find a real match, this section switches to Fed rate decisions — a category where both platforms tend to cover the same specific FOMC meetings with very similar phrasing.

In [97]:
# --- Kalshi: Fed rate decisions live under the KXFED series ---
# Pull a bigger pool (200) since KXFED spans many future meeting dates,
# then filter down to one specific meeting so we're comparing the same event.
kalshi_fed_raw = requests.get(
    "https://api.elections.kalshi.com/trade-api/v2/markets",
    params={"status": "open", "limit": 200, "series_ticker": "KXFED", "mve_filter": "exclude"}
).json()["markets"]

TARGET_MONTH = "Jul"
TARGET_YEAR = "2026"

kalshi_fed_titles = [
    m["title"] for m in kalshi_fed_raw
    if TARGET_MONTH in m["title"] and TARGET_YEAR in m["title"]
]

print(f"Kalshi Fed markets ({TARGET_MONTH} {TARGET_YEAR} meeting): {len(kalshi_fed_titles)}")
for t in kalshi_fed_titles[:10]:
    print(f"- {t}")

print()

# --- Polymarket: sort by volume so high-traffic Fed markets surface,
# then filter to the same target month/year meeting ---
poly_fed_events = requests.get(
    "https://gamma-api.polymarket.com/events",
    params={
        "active": "true",
        "closed": "false",
        "limit": 100,
        "order": "volume24hr",
        "ascending": "false",
    }
).json()

poly_fed_titles = []
for e in poly_fed_events:
    title_lower = e.get("title", "").lower()
    if "fed" in title_lower or "fomc" in title_lower:
        for m in e.get("markets", []):
            question = m["question"]
            # keep only markets mentioning the same target month/year
            if TARGET_MONTH.lower() in question.lower() and TARGET_YEAR in question:
                poly_fed_titles.append(question)

print(f"Polymarket Fed markets ({TARGET_MONTH} {TARGET_YEAR} meeting): {len(poly_fed_titles)}")
for t in poly_fed_titles[:10]:
    print(f"- {t}")

Kalshi Fed markets (Jul 2026 meeting): 11
- Will the upper bound of the federal funds rate be above 5.25% following the Fed's Jul 29, 2026 meeting?
- Will the upper bound of the federal funds rate be above 5.00% following the Fed's Jul 29, 2026 meeting?
- Will the upper bound of the federal funds rate be above 4.75% following the Fed's Jul 29, 2026 meeting?
- Will the upper bound of the federal funds rate be above 4.50% following the Fed's Jul 29, 2026 meeting?
- Will the upper bound of the federal funds rate be above 4.25% following the Fed's Jul 29, 2026 meeting?
- Will the upper bound of the federal funds rate be above 4.00% following the Fed's Jul 29, 2026 meeting?
- Will the upper bound of the federal funds rate be above 3.75% following the Fed's Jul 29, 2026 meeting?
- Will the upper bound of the federal funds rate be above 3.50% following the Fed's Jul 29, 2026 meeting?
- Will the upper bound of the federal funds rate be above 3.25% following the Fed's Jul 29, 2026 meeting?
- Wi

In [98]:
kalshi_fed_embeddings = model.encode(kalshi_fed_titles)
poly_fed_embeddings = model.encode(poly_fed_titles)

fed_cosine_scores = util.cos_sim(kalshi_fed_embeddings, poly_fed_embeddings)

fed_matches = []
for i in range(len(kalshi_fed_titles)):
    for j in range(len(poly_fed_titles)):
        fed_matches.append((fed_cosine_scores[i][j].item(), kalshi_fed_titles[i], poly_fed_titles[j]))

fed_matches.sort(reverse=True, key=lambda x: x[0])

print("=== Top 10 Fed matches ===\n")
for score, k_title, p_title in fed_matches[:10]:
    print(f"[{score:.3f}]")
    print(f"  Kalshi:     {k_title}")
    print(f"  Polymarket: {p_title}\n")

=== Top 10 Fed matches ===

[0.716]
  Kalshi:     Will the upper bound of the federal funds rate be above 3.25% following the Fed's Jul 29, 2026 meeting?
  Polymarket: Will the Fed increase interest rates by 25 bps after the July 2026 meeting?

[0.711]
  Kalshi:     Will the upper bound of the federal funds rate be above 4.25% following the Fed's Jul 29, 2026 meeting?
  Polymarket: Will the Fed increase interest rates by 25 bps after the July 2026 meeting?

[0.711]
  Kalshi:     Will the upper bound of the federal funds rate be above 5.25% following the Fed's Jul 29, 2026 meeting?
  Polymarket: Will the Fed increase interest rates by 25 bps after the July 2026 meeting?

[0.705]
  Kalshi:     Will the upper bound of the federal funds rate be above 5.00% following the Fed's Jul 29, 2026 meeting?
  Polymarket: Will the Fed increase interest rates by 25 bps after the July 2026 meeting?

[0.703]
  Kalshi:     Will the upper bound of the federal funds rate be above 3.50% following the Fed's 

In [99]:
TOP_N_FED = 10
fed_candidates = fed_matches[:TOP_N_FED]

print(f"Verifying top {TOP_N_FED} Fed matches with Groq...\n")

fed_verified = []
for score, k_title, p_title in fed_candidates:
    is_match, reason = verify_match(k_title, p_title)
    fed_verified.append((score, k_title, p_title, is_match, reason))
    status = "CONFIRMED MATCH" if is_match else "FALSE POSITIVE"
    print(f"[{score:.3f}] {status}")
    print(f"  Kalshi:     {k_title}")
    print(f"  Polymarket: {p_title}")
    print(f"  Reason:     {reason}\n")

confirmed_fed = [r for r in fed_verified if r[3]]
print(f"Confirmed true positives: {len(confirmed_fed)} / {TOP_N_FED}")

Verifying top 10 Fed matches with Groq...

[0.716] FALSE POSITIVE
  Kalshi:     Will the upper bound of the federal funds rate be above 3.25% following the Fed's Jul 29, 2026 meeting?
  Polymarket: Will the Fed increase interest rates by 25 bps after the July 2026 meeting?
  Reason:     The questions are about different aspects of the Fed's decision, with Question A focusing on the upper bound of the federal funds rate and Question B focusing on the size of the interest rate increase.

[0.711] FALSE POSITIVE
  Kalshi:     Will the upper bound of the federal funds rate be above 4.25% following the Fed's Jul 29, 2026 meeting?
  Polymarket: Will the Fed increase interest rates by 25 bps after the July 2026 meeting?
  Reason:     The questions are about different aspects of the Fed's decision, with Question A focusing on the upper bound of the federal funds rate and Question B focusing on the size of the interest rate increase.

[0.711] FALSE POSITIVE
  Kalshi:     Will the upper bound of 

## 9: Conclusions so far

This notebook set out to test one core idea: can semantic similarity (a shared sentence-transformer + cosine similarity) reliably find the same real-world event listed on two different prediction market platforms?

**Finding 1 — Combo markets are a real data quality problem.** Kalshi's `/markets` endpoint mixes ordinary single-outcome markets with multivariate combo markets (parlays bundling several unrelated events into one ticket). These have to be filtered out before matching, using `mve_filter=exclude` or checking `mve_selected_legs`, or they pollute the candidate pool with noise.

**Finding 2 — Category/tag filtering is not obvious and is platform-specific.** Kalshi's `category` field lives on `/events`, not `/markets`. Polymarket uses a numeric `tag_id`, not a text category name, and even tag-based filtering can miss high-volume categories (Fed markets) if the pulled page happens not to include them — sorting by `volume24hr` was a more reliable way to surface them than guessing tags.

**Finding 3 — Cosine similarity produces confident, wrong matches when two different races share a sentence template.** The DNC Chair vs. 2028 nomination test is the clearest example: 50 out of 50 top-ranked "matches" scored 0.65–0.85 similarity, and every single one was a false positive, because "Will [Name] [do political thing]?" is a shared structure across two genuinely different elections. A threshold-only system would have confidently flagged all 50 as real.

**Finding 4 — An LLM verification pass reliably catches this failure mode.** Sending each candidate pair to Groq with a same-outcome question caught all 50 DNC/nomination false positives, each with a specific, correct, non-generic reason naming the two different roles involved.

**Finding 5 — Text-only verification has its own limit: it can't do numeric reasoning.** On Fed rate markets aligned to the same meeting date (July 2026), Kalshi asks about an absolute rate threshold ("above 4.25%") while Polymarket asks about a rate change ("+25 bps"). These *can* be the same underlying bet, but only if you know the current fed funds rate going into the meeting — information neither the embedding model nor a text-only LLM prompt has access to. Groq correctly declined to guess rather than hallucinating a match, which is the right failure mode for a system whose output could inform real trades.

**Where this leaves HIVE:** a two-stage pipeline (embeddings for candidate generation, LLM for verification) is a solid foundation, but it is not sufficient on its own for every market type. Threshold/numeric markets like Fed rate decisions likely need a third stage: structured parsing of the specific claim (threshold value, direction, current baseline) rather than relying on semantic similarity or free-text LLM judgment alone. That is a natural next milestone rather than a flaw in what has been built so far.